In [8]:
import matplotlib.pyplot as plt
import pandas as pd
from lmfit import Model
%run functions_dbs.py

fs = 10
%matplotlib

Using matplotlib backend: MacOSX


In [9]:
file = 'Complete_T1.xlsx'
ddata = pd.read_excel(file, sheet_name=None)

# get the EP data (ideally all profiles)
ls_ep = list()
[ls_ep.append(i) for i in ddata.keys() if 'EP' in i or 'Ep' in i or 'ep' in i]
key = None
for x in ls_ep:
    if 'all' in x:
        key = x
if key:
    pass
else:
    key = ls_ep[0]

df_ep = ddata[key]

# get core - sample order
order = df_ep[['Nr','Core']].drop_duplicates().values


# split into cores
gb = df_ep.groupby(by='Core')
dic_ep = dict(map(lambda x: (x, gb.get_group(x)), gb.groups))

# split one core into sample-Nr
for c in dic_ep.keys():
    gp = dic_ep[c].groupby(by='Nr')
    dic_epS = dict(map(lambda x: (x, gp.get_group(x)), gp.groups))
    dic_ep[c] = dic_epS

for c in dic_ep.keys():
    for s in dic_ep[c].keys():
        d = dic_ep[c][s][['Depth', 'EP_mV']].set_index('Depth')
        dic_ep[c][s] = d

df_ep = pd.concat(dict(map(lambda c: (c, pd.concat(dic_ep[c], axis=1)), dic_ep.keys())), axis=1)

# bring to correct order
ls = list()
[ls.append(o[1]) for o in order if o[1] not in ls]

new_cols = df_ep.columns.reindex(ls, level=0)
df_ep = df_ep.reindex(columns=new_cols[0])

In [10]:
# 1st datapoint --> the most positive one
ls_1st = list()
for c in df_ep.columns:
    ls_1st.append(df_ep[c].loc[df_ep[c].dropna().index[0]])

df = pd.DataFrame(ls_1st, columns=['1st datapoint'])

# --------------------------------------------------
# remove outlier --> z score
q75,q25 = np.percentile(df, q=[75,25])
intr_qr = q75-q25

max = q75+(1.5*intr_qr)
min = q25-(1.5*intr_qr)

df[df < min] = np.nan
df[df > max] = np.nan

df = df.dropna()
df

,1st datapoint
1,34.382629
2,34.489746
3,33.480072
4,33.787689
5,33.512726
6,33.409424
7,33.330536
8,34.272003
9,33.107910
10,32.950745


In [11]:
# polynomial fit of 2nd order: ax**2 + bx + c = y
paraFit = np.polyfit(df.index, df['1st datapoint'].to_numpy(), deg=2)
paraFit

array([ 7.68353480e-03, -2.33361564e-01,  3.46279224e+01])

In [12]:
# drift correction
df_ep_corr = df_ep.copy()

i, i0 = 1, 1
df_ep_corr[35][15].loc[:] = (df_ep[35][13] + (paraFit[0]*(i**2 - i0**2) + paraFit[1]*(i - i0))).values

In [17]:
plt.plot(df_ep_corr[35][15].dropna()['EP_mV'].to_numpy(), df_ep_corr[35][15].dropna().index)